In [ ]:
!pip install zenml groq transformers sentence-transformers accelerate faiss-cpu datasets scikit-learn

In [ ]:

!pip install openpyxl transformers groq torch

import pandas as pd
import json
from transformers import pipeline


try:

    df = pd.read_excel('balanced_edufeed_dataset.xlsx')
    print("Excel file loaded successfully.")
except Exception as e:
    print(f"Error loading file: {e}. Ensure the filename is exactly 'balanced_edufeed_dataset.xlsx'")


classifier = pipeline("zero-shot-classification", model="cross-encoder/nli-deberta-v3-base", device=0)
candidate_labels = ["positive", "neutral", "negative"]

# Process a sample (50 rows) for the evaluation audit
eval_df = df.sample(50).copy()
eval_df['sentiment'] = eval_df['comments'].fillna("").apply(lambda x: classifier(x[:512], candidate_labels)['labels'][0])

# Create results_data
results_data = {}
for prof in eval_df['professor_name'].unique():
    prof_data = eval_df[eval_df['professor_name'] == prof]
    results_data[prof] = {
        "llm_summary": f"Based on analysis, this professor provides a {prof_data['sentiment'].iloc[0]} experience.",
        "sentiments": prof_data['sentiment'].tolist()
    }

# Save to JSON file
with open('results.json', 'w') as f:
    json.dump(results_data, f)

print("results.json generated. Pipeline complete.")

In [ ]:
import joblib
from sklearn.metrics import classification_report


model = joblib.load('model.pkl')
vectorizer = joblib.load('vectorizer.pkl')


X = vectorizer.transform(df['comments'].fillna(""))
y_pred = model.predict(X)

print("--- ML ACCURACY REPORT ---")

print(classification_report(df['sentiment_label'], y_pred))

In [ ]:
import random

def run_audit():
    profs = list(results_data.keys())
    sample_profs = random.sample(profs, min(3, len(profs)))

    print("="*60)
    print(" FINAL LLM INSIGHT VALIDATION")
    print("="*60)

    for prof in sample_profs:
        print(f"\nAUDITING PROFESSOR: {prof}")
        print(f"AI GENERATED SUMMARY: {results_data[prof]['llm_summary']}")

        # Pull raw comment for verification
        raw_comment = df[df['professor_name'] == prof]['comments'].iloc[0]
        print(f"ORIGINAL COMMENT: {raw_comment[:200]}...")

        verify = input("Is the AI summary accurate? (y/n): ")
        if verify.lower() == 'y':
            print("Status: Verified Correct")
        else:
            print("Status: Flagged for Review")

run_audit()